# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]


Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the provided complaints, appear to be:\n\n- Dealing with lender or servicer misconduct, including errors in loan balances, misapplied payments, and wrongful denials of payment plans.\n- Incorrect information on credit reports, such as falsely reported delinquencies or misreported account statuses.\n- Problems with payment handling, such as restrictions on applying extra funds to principal, or difficulties in making payments.\n- Issues related to loan management, including unnotified transfers between loan servicers, mismanagement of forbearances, and improper handling of loan data.\n- Disputes over loan balances, interest calculations, and loan mishandling, often leading to credit damage or financial hardship.\n- Struggles with loan forgiveness, cancellation, or discharge processes, especially for older or mismanaged loans.\n- Discrepancies and inaccuracies in loan account information and documentation.\n\nIn summary, the most common issue s

In [12]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided data, yes, some complaints did not get handled in a timely manner. Specifically, there are complaints where the response status indicates delays:\n\n- A complaint received on 03/28/25 by MOHELA, where the response was marked as "No" for timely response, indicating it was not handled in a timely manner.\n- Multiple complaints, such as those regarding unresolved issues with loan account corrections, payment application, and information reporting, have been open for over a year or several months without resolution, suggesting delays in handling.\n\nIn contrast, some complaints received responses marked as "Yes" for timely response, indicating they were handled promptly. Nonetheless, the presence of complaints marked "No" for timely response confirms that not all complaints were addressed promptly.\n\nTherefore, the answer is: **Yes, some complaints did not get handled in a timely manner.**'

In [13]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans largely due to issues such as lack of clear communication from loan servicers, unexpected resumption of payments, and increased interest that negated progress on repayments. Many borrowers were unaware of when their payments were supposed to restart, were not properly notified of transfers between loan servicers, or faced difficulties in understanding the status and details of their loans. Additionally, some borrowers found that their payments primarily covered interest due to how their payments were applied, making it difficult to reduce the principal. Financial hardships, stagnating wages, and misinformation about loan terms and forgiveness programs also contributed to the inability to repay loans effectively.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with student loan complaints appears to be problems related to dealing with lenders or servicers, particularly issues with the handling of payments, fees, and accurate information about the loan. Specifically, complaints often involve:\n\n- Disagreements over fees charged\n- Difficulties applying payments correctly, especially to principal versus interest\n- Receiving wrong or misleading information about loan balances, terms, or repayment plans\n- Issues with loan servicers not addressing or resolving these problems adequately\n\nThis suggests that the most common issue is related to "Dealing with your lender or servicer," often involving mismanagement, misinformation, or disputes over fees and payment applications.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all the complaints listed include responses labeled as "Timely response?" marked as "Yes." Therefore, it appears that any complaints that were handled were addressed in a timely manner.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n- Problems with managing or understanding their payment plans, such as being steered into the wrong types of forbearances or having automatic payments unenrolled without their knowledge.\n- Lack of proper communication from the loan servicers, leading borrowers to be unaware of their account status, payment requirements, or important updates.\n- Errors or issues with the payment processing system, causing payments to be reversed or not processed correctly.\n- Discrepancies in account information, such as incorrect bank details, which hinder payment processing.\n- Administrative mistakes, like transferring loans between companies without proper notification, which resulted in missed payments and negative impacts on credit scores.\n- Inability to resolve issues promptly due to poor customer service, leading to unresolved problems and accumulating interest or bills.\n\nIn summary, failures to pay back loans often st

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.
####✅ Answer: What was the weather in Charlotte yesterday? Because BM25 works better when the query need retrival based on precise keyword matches  vs abstract and semantic generalization needs

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [95]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [96]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [97]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, a common issue with loans appears to be errors and mismanagement by loan servicers, such as incorrect loan balances, misapplied payments, lack of communication, and mishandling of loan information. Many complaints involve disputes over loan amounts, interest calculations, unauthorized transfers, and mishandling of personal data. Therefore, a frequent and significant issue is **errors and mismanagement by loan servicers leading to incorrect loan information and disputes.**'

In [98]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that at least one complaint was handled promptly with a response marked as "Yes" for timely response. For example, the complaint submitted to Maximus Federal Services, Inc. was responded to in a timely manner, and the issue was marked as "Closed with explanation." \n\nHowever, the complaint from the same individual regarding a year-long delay and lack of resolution concerning account review and loan information indicates that some issues did not get resolved in a timely manner over an extended period. The complaint from the other individual related to payment processing issues was also marked "timely response," but the ongoing nature of the problems suggests that resolution delays occurred in some cases.\n\nIn summary:\n- Yes, some complaints were handled in a timely manner.\n- No, some complaints, particularly regarding unresolved issues over more than a year, indicate that certain complaints did not get handled in a timely fashion.\n\nThe

In [99]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily because of a lack of clear communication, insufficient information about their loan terms, and the complicated nature of loan interest accrual. Many borrowers were unaware that they needed to repay their loans or did not receive proper notifications when their loans were taken over or transferred between lenders. Additionally, options like forbearance or deferment led to continued interest accumulation, which increased the total amount owed over time. Borrowers also faced difficulties in managing the growing interest, especially when payments were insufficient or when they faced financial hardships, making it virtually impossible to pay off the loans within the expected timeframe. Overall, issues such as misinformation, lack of transparency, and the complexity of loan servicing contributed to borrowers' inability to repay their loans effectively."

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [100]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [101]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [102]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans include:\n\n- Errors and inaccuracies in loan balances, interest calculations, and account status reports.\n- Problems with loan servicers providing bad information, misapplying payments, or mishandling loan details.\n- Difficulties with payment management, such as inability to apply extra payments toward principal, or payments being applied improperly.\n- Unnotified transfers of loan servicing and updates that impact credit reports and payment obligations.\n- Discrepancies in reported loan status on credit reports, such as incorrect delinquencies or missing payment history.\n- Issues related to loan forgiveness, cancellation, or discharge due to mismanagement or legal disputes.\n- Unauthorized sharing of personal information or improper use of borrower data.\n- Aggressive collection actions and improper communication tactics.\n\nOverall, a recurring theme is that borrowers frequently face inaccuracies, miscommunications

In [103]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, according to the provided complaints, several complaints indicate that they were not handled in a timely manner. Specifically:\n\n- One complaint from 04/18/25 was marked as "Timely response?": No, implying it was not handled promptly.\n- Multiple complaints from late April and early May 2025 explicitly mention delays, unanswered calls, or complaints still unresolved after extended periods, such as over a year or several months.\n- For example, a complaint submitted on 04/04/25 notes attempts to resolve issues over nearly two and a half years with ongoing delays and unresponsive service.\n- Multiple complaints also report that the companies failed to respond or resolve issues within the mandated timeframes.\n\nTherefore, yes, there are complaints that did not get handled in a timely manner.'

In [105]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People often failed to pay back their loans due to a variety of systemic and service-related issues, including:\n\n1. Lack of clear information about repayment options, interest accrual, and programs like income-driven repayment or loan forgiveness, leading borrowers to rely on forbearance and incur more interest.\n2. Being steered into long-term forbearances instead of more manageable repayment plans, resulting in interest accumulation and ballooning balances.\n3. Poor communication from lenders and servicers, with some borrowers reporting not being notified of overdue payments or default status.\n4. Errors or misinformation reported on credit reports due to mishandling or inadequate servicing, adversely affecting credit scores.\n5. Mismanagement and wrongful reporting of loan delinquency, sometimes without proper notice or in violation of federal regulations.\n6. Inadequate customer support and guidance, leading to borrowers being unable to negotiate or understand their repayment op

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.
###✅ Answer: I can't expect the user to be trained on prompting, and my app should handle queries written in all type of prose and loaded with errors. query reformulation would allow the retriver to retrive relevant information that semantically matches several reforumulations of the original query (ideally matching the actul intent of the query better). It increases retrival diversity, by retriving context that may come from lexically variant sources.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [106]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [109]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [110]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [111]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [112]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [113]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be related to problems with loan servicing, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and misconduct by loan servicers. Additionally, issues like incorrect information on credit reports, unfair or unjustified interest rate increases, and difficulties in verifying or legally substantiating debt are frequently mentioned.'

In [114]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all the complaints listed were handled but some were not handled in a timely manner. Specifically, the complaints against MOHELA (rows 441 and 84) were marked as "Timely response?": "No," indicating that they did not receive a prompt response. \n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [115]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors such as experiencing severe financial hardship, lack of proper information or guidance, and issues related to the management and transparency of the loan process. For example, some borrowers faced unexpected payment demands during their grace period or after their schools closed and failed to receive clear communication about payment obligations. Others were unable to secure employment or manage their finances due to misleading or incomplete information about the value of their education, the stability of their institutions, or the long-term consequences of borrowing. Additionally, there are cases where issues with loan servicing, such as misreported payments, failure to notify borrowers, or problems related to the legitimacy of the debt, contributed to difficulties in repayment.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [116]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [117]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [118]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the context provided, appears to be dealing with the mismanagement and inaccuracies related to loan servicing. Specific recurring issues include:\n\n- Errors in loan balances and interest calculations.\n- Receiving bad or false information about loan terms, balances, or interest accrual.\n- Problems with how payments are being handled or applied, often leading to increased balances due to interest accumulation.\n- Unfair or predatory practices, such as being unable to apply extra payments toward principal, or being misclassified in loan programs.\n- Challenges with loan transfers, improper end of deferments, or incorrect classification of loan types.\n- Discrepancies and errors in credit reporting linked to loans.\n\nOverall, the overarching issue seems to be the improper handling, inaccuracy, or misreporting of loan details by servicers, which leads to financial and credit impacts on borrowers.'

In [119]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, it appears that several complaints were not handled in a timely manner. For example:\n\n- Complaint ID: 12935889, submitted to MOHELA on 03/25/25, was marked as "No" for timely response.\n- Complaint ID: 12654977, submitted to MOHELA on 03/25/25, was marked as "No" for timely response.\n- Complaint ID: 12914633, submitted to Higher Education Servicing Corporation (IL) on 04/10/25, was marked as "Yes" for timely response.\n- Complaint ID: 13056764, submitted to EdFinancial Services on 04/18/25, was marked as "Yes."\n- Complaint ID: 12823876, submitted to EdFinancial Services on 04/04/25, was marked as "Yes."\n- Complaint ID: 13062402, submitted to Nelnet on 04/18/25, was marked as "Yes."\n- Complaint ID: 13160766, submitted to Maximus Federal Services (MI) on 04/24/25, was marked as "Closed with explanation," which may reflect handling, but the status varies.\n\nThe key indication in the data is that some complaints received a response marked as "

In [120]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a combination of factors such as:\n\n1. Lack of clear and timely communication from lenders or servicers about payment obligations, changes in loan status, or available repayment options.\n2. Mismanagement or misreporting of loan information, leading to incorrect delinquencies or balances being reported on credit reports.\n3. Accumulation of interest during forbearance or deferment periods, which increased the total amount owed and made repayment more difficult.\n4. Lack of awareness or understanding of complex loan terms, including interest accrual, consolidation, or income-driven repayment programs.\n5. Financial hardships such as unemployment, low income, unexpected life events (e.g., health issues, homelessness), or economic downturns, which limited borrowers' ability to make payments.\n6. Servicer misconduct, including steering borrowers into forbearances rather than repayment plans, failing to notify borrowers about starting

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [121]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [122]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [124]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [125]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [127]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [128]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data, the most common issues with loans appear to involve problems related to communication and documentation, such as:\n\n- Trouble with how payments are being handled or processed\n- Disputes over incorrect account status or reporting (e.g., loans in default when borrower states they are not)\n- Poor or inconsistent communication from lenders or servicers\n- Issues with how payment plans or re-amortizations are applied\n- Concerns over improper or illegal reporting and data breaches\n\nSpecifically, many complaints highlight difficulties in obtaining clear information, inconsistent payment processing, and errors in account status which can significantly impact borrowers’ credit reports and financial standing.\n\nTherefore, the most common issue with loans, as reflected in these complaints, is **"Problems with loan servicing and communication, including incorrect account status, payment processing errors, and inadequate transparency."**\n\nIf you need

In [129]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided complaints, several complaints indicate that complaints were not handled in a timely manner. Specifically, the complaint about the transfer of an account to Nelnet involved the company not responding to certified mail despite acknowledging receipt, which suggests a delay or failure in handling the complaint timely. Additionally, multiple complaints are characterized as "Closed with explanation," which can sometimes imply they were unresolved or not handled promptly. However, all responses explicitly state "Yes" to "Timely response?" indicating that, according to the records, responses were made within the expected timeframe for those cases.\n\nIn summary, while most complaints received timely responses, there are indications that some complaints, such as the one involving the transfer and misconduct, were not adequately addressed in a timely manner.'

In [130]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons based on the complaints provided. Some common reasons include:\n\n1. **Lack of Transparency and Poor Communication:** Borrowers experienced difficulty obtaining clear information about their loan status, forbearance periods, or repayment terms, which led to confusion and default. For example, one complaint mentioned receiving bad information about their loan, with no written confirmation, and facing long waits when seeking assistance.\n\n2. **Problems with Loan Processing and Documentation:** Borrowers encountered issues such as missing documentation, delays, or rejection of submitted paperwork necessary for loan forgiveness or correct loan status, causing frustration and potential default.\n\n3. **Increased Payment Amounts After Forbearance:** When the COVID-19 forbearance ended, some borrowers faced significantly higher payments due to lack of timely re-amortization, which they could not afford, resulting in missed payments.\

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?
### ✅✅✅ Answer: If questions are repetitive, semantic chunking will merge the questions into similar chunks, losing distinction of questions, and providing redundant, blended, generic answers. One way is to not use semantic chunking for questions in FAQ, but alternatively, reduce the chunk size with clear seperators to presever the units within each FAQ. Second option is to treat each questions as a separate chunk. 

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [56]:
#NLTK Import To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/chrag/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/chrag/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [57]:
### Step 0: Dependencies & imports

# included in pyproject.toml file
# "numpy>=2.2.2",
# "ragas==0.3.0",
# "rapidfuzz"
# "langchain-core",
# "langchain-community",
# "langsmith",
#  "tqdm", 


#Setting up the LLM and embedding model and generator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [ ]:
#Step 1 Create a "golden dataset" a.k.a synthetic test data
# This will generate our knowledge graph under the hood and generate our personas and scenarios to construct our queries

from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(loan_complaint_data, testset_size=20)

Applying SummaryExtractor:   0%|          | 0/539 [00:00<?, ?it/s]

Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt summary_extractor_prompt failed to parse output: The output parser failed to parse the output including retries.
unable to apply transformation: The output parser failed to parse the output including retries.


Applying CustomNodeFilter:   0%|          | 0/825 [00:00<?, ?it/s]

Node 17498427-14a3-4ecf-ba44-8f18a4a3d90d does not have a summary. Skipping filtering.
Node be292c3f-bb84-4c03-85b6-908ca5e6349d does not have a summary. Skipping filtering.
Node e4201186-7244-4c32-a5d0-1a2b86289202 does not have a summary. Skipping filtering.
Node 35c4d72c-de16-4a0b-a0f4-8e97cdc59d63 does not have a summary. Skipping filtering.
Node 815fe96e-fc3f-4033-bf8c-98172e3ba65d does not have a summary. Skipping filtering.
Node 3acb6c4e-3718-4328-8222-2f8dfbc5caff does not have a summary. Skipping filtering.
Node 7d4d94bd-1160-4f1b-89a8-527a6129e22d does not have a summary. Skipping filtering.
Node 9bd79867-ea1e-4ea7-a9f8-87c97fbfe3ac does not have a summary. Skipping filtering.
Node 7c24c4be-fc55-4d46-80d8-f30d1c1d9111 does not have a summary. Skipping filtering.
Node e21f9ef1-6781-41ef-b643-b514f30d9ed0 does not have a summary. Skipping filtering.
Node 785160d0-1cc3-4b30-9eab-aa0a5fc48f40 does not have a summary. Skipping filtering.
Node ffcd029f-cd7a-4812-8ac4-2f9ed99c8002 d

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/2189 [00:00<?, ?it/s]

...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

unable to apply transformation: Node 65643873-e16c-4de0-91bc-8278349d3598 has no summary_embedding
unable to apply transformation: Node 717a6fff-a904-4c15-bfb6-35bc9924f6a6 or 437d76ee-f354-4737-a3c4-1db2ccba0e6c has no entities


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/20 [00:00<?, ?it/s]

In [62]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,When did the COVID-19 forbearance program end?,[The federal student loan COVID-19 forbearance...,The federal student loan COVID-19 forbearance ...,single_hop_specifc_query_synthesizer
1,Aidvantage why my payment wrong and how they c...,[I submitted my annual Income-Driven Repayment...,Aidvantage assigned me a payment amount that i...,single_hop_specifc_query_synthesizer
2,How does FERPA relate to my student loan debt ...,[My personal and financial data was compromise...,My personal and financial data was compromised...,single_hop_specifc_query_synthesizer
3,Whaat is the deal with Nelnet and my student l...,"[According to Studentaid.gov, Im to get an ema...","According to the context, the person is confus...",single_hop_specifc_query_synthesizer
4,What does 15 U S C 16811 say about how credit ...,[I am writing to formally dispute inaccurate i...,15 U.S.C. 16811 requires credit reporting agen...,single_hop_specifc_query_synthesizer
5,How can I get Aid Avantage to remove the past ...,[I am devastated. I would like to report a sit...,The individual is seeking to have Aid Avantage...,single_hop_specifc_query_synthesizer
6,Did the Department of Education get my info?,"[On XXXX XXXX XXXX, XXXX XXXX instructed his t...","On XXXX XXXX XXXX, XXXX XXXX instructed his te...",single_hop_specifc_query_synthesizer
7,Why EdFinancials not accept my docs?,[I have provided documentation relating to my ...,The documentation relates to a $5000.00 teache...,single_hop_specifc_query_synthesizer
8,How does FERPA relate to my personal data bein...,[My personal and financial data was compromise...,The context states that personal and financial...,single_hop_specifc_query_synthesizer
9,How does the Privacy Act of 1974 relate to my ...,[I am writing to formally dispute my XXXX XXXX...,The Privacy Act of 1974 is a federal law that ...,single_hop_specifc_query_synthesizer


In [151]:
#all imports
import copy
import time
import pandas as pd
from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from langchain_core.runnables import Runnable

In [ ]:
#setting up LangSmith api and project   
import os
import getpass
from langsmith import Client
from langsmith.utils import traceable

os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("🔐 LangSmith API Key: ") # Securely input your LangSmith API key
os.environ["LANGCHAIN_PROJECT"] = "09_Advanced_Retrieval" # Set project name (must be via environment var)
os.environ["LANGCHAIN_TRACING_V2"] = "true" # enables LangSmith's latest tracing system, known as "Tracing V2"

client = Client() # Initialize LangSmith client

In [ ]:
#defines evaluate function for retriever metrics

@traceable(run_type="evaluation", name="Evaluate Retriever", tags=["retriever"])
def evaluate_retriever(
    dataset,
    retriever_chain: Runnable,
    delay_sec: float = 1.0,
    model_name: str = "gpt-4.1-mini",
    timeout: int = 360,
    verbose: bool = False,
    return_dataset: bool = False,
):
    eval_dataset = copy.deepcopy(dataset)

    for i, test_row in enumerate(eval_dataset):
        user_question = test_row.eval_sample.user_input

        if verbose:
            print(f"[{i+1}/{len(eval_dataset)}] Querying retriever: {user_question}")

        result = retriever_chain.invoke({"question": user_question})

        test_row.eval_sample.retrieved_contexts = [
            doc.page_content for doc in result["context"]
        ]
        test_row.eval_sample.response = " "

        time.sleep(delay_sec)

    df = pd.DataFrame([row.eval_sample.to_dict() for row in eval_dataset])
    df["response"] = df["response"].fillna(" ")
    ragas_dataset = EvaluationDataset.from_pandas(df)

    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model=model_name))
    run_config = RunConfig(timeout=timeout)

    retriever_metrics = [
        LLMContextRecall(),        # Measures how much of the relevant context (needed to answer the query) is retrieved
        ContextEntityRecall(),     #  Measures whether key entities from the gold/reference answer are present in the retrieved context.
        ContextPrecision(),        #  easures the proportion of relevant chunks in the retrieved contexts
        NoiseSensitivity(),         #  easures how often a system makes errors by providing incorrect responses
    ]

    results = evaluate(
        dataset=ragas_dataset,
        metrics=retriever_metrics,
        llm=evaluator_llm,
        run_config=run_config,
    )

    return (results, eval_dataset) if return_dataset else results

In [153]:
 # Copy dataset to avoid modifying original
eval_dataset = copy.deepcopy(dataset)

In [ ]:
# Call the evaluator function for naive retriever
naive_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=naive_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(naive_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


### {'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for bm25
bm25_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=bm25_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(bm25_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[37]: TimeoutError()


{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


### {'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for MultiQueryRetriever
MultiQueryRetriever_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=multi_query_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(MultiQueryRetriever_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[15]: TimeoutError()
Exception raised in Job[25]: TimeoutError()


{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


###{'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for Parent Document Retriever
parent_document_retriever_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=parent_document_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(parent_document_retriever_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


###{'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for Ensemble Retriever
ensemble_retrieval_chain_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=ensemble_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(ensemble_retrieval_chain_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[9]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[3]: TimeoutError()


{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


###{'context_recall': 0.9500, 'context_entity_recall': 0.5708}